In [1]:
import pandas as pd
from pathlib import Path

In [2]:
file_path = Path("RASFF_raw_data_EU.csv")

def perform_eda(file_path: Path) -> pd.DataFrame:
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin1"]
    for enc in encodings:
        try:
            df = pd.read_csv(file_path, encoding=enc, low_memory=False)
            print(f"Loaded {file_path.name} with encoding={enc}")
            return df
        except UnicodeDecodeError:
            continue
        except pd.errors.ParserError:
            try:
                df = pd.read_csv(
                    file_path,
                    encoding=enc,
                    engine="python",
                    on_bad_lines="skip",
                )
                print(
                    f"Loaded {file_path.name} with encoding={enc}, engine=python, on_bad_lines=skip "
                    "(some malformed rows may be skipped)"
                )
                return df
            except Exception:
                continue

    # Last fallback: keep pipeline alive by replacing undecodable bytes.
    df = pd.read_csv(
        file_path,
        encoding="utf-8",
        encoding_errors="replace",
        engine="python",
        on_bad_lines="skip",
    )
    print(
        f"Loaded {file_path.name} with encoding=utf-8 (invalid bytes replaced), "
        "engine=python, on_bad_lines=skip"
    )
    return df

df = perform_eda(file_path)
print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
df.head(3)

Loaded RASFF_raw_data_EU.csv with encoding=utf-8, engine=python, on_bad_lines=skip (some malformed rows may be skipped)
Rows: 29,624 | Columns: 14


,reference,category,type,subject,date,notifying_country,classification,risk_decision,distribution,forAttention,forFollowUp,operator,origin,hazards
0,2026.2434,"dietetic foods, food supplements and fortified...",food,Unlabelled allergen soja in whey protein isolate,20-03-2026 18:01:28,Switzerland,alert notification,serious,Switzerland,NaN,Germany,"Germany,Switzerland",Germany,soya presence - {allergens}
1,2026.2433,"nuts, nut products and seeds",food,Aflatoxin B1 and totoal on Groundnuts from Arg...,20-03-2026 17:55:22,Netherlands,information notification for attention,serious,Netherlands,INFOSAN,NaN,"Argentina,Netherlands",Argentina,"Aflatoxin B1 - {mycotoxins},aflatoxin total ..."
2,2026.2432,fruits and vegetables,food,Exceeded MRL pesticide Oxamyl in oranges from ...,20-03-2026 17:51:42,Romania,border rejection notification,potentially serious,NaN,INFOSAN,NaN,"Egypt,Romania",Egypt,oxamyl unauthorised substance - {pesticide re...


## Focused EDA: Vietnam, India, Ecuador (SPS/TBT)
This section filters records to the 3 countries and categorizes rejection reasons using keyword rules for SPS and TBT.

In [11]:
import re

TARGET_COUNTRIES = {"vietnam", "india", "ecuador"}
START_DATE = pd.Timestamp("2020-03-30")
END_DATE = pd.Timestamp("2025-12-31")

SPS_KEYWORDS = [
    r"\bsps\b",
    r"sanitary",
    r"phytosanitary",
    r"pesticide",
    r"residue",
    r"mr\s*l",
    r"microbiolog",
    r"salmonella",
    r"listeria",
    r"aflatoxin",
    r"contamin",
    r"pathogen",
    r"veterinary",
    r"animal health",
    r"plant health",
    r"food safety",
    r"mycotoxin",
]

TBT_KEYWORDS = [
    r"\btbt\b",
    r"technical barrier",
    r"technical regulation",
    r"label",
    r"labelling",
    r"packag",
    r"certificate",
    r"documentation",
    r"traceability",
    r"standard",
    r"marking",
    r"conformity",
    r"compliance",
    r"composition",
    r"additive",
    r"quality requirement",
]

def pick_first_column(columns: list[str], include_terms: list[str]) -> str | None:
    for col in columns:
        col_norm = col.lower().strip()
        if any(term in col_norm for term in include_terms):
            return col
    return None

def pick_country_column(columns: list[str]) -> str | None:
    col_map = {c.lower().strip(): c for c in columns}

    # Prefer origin/source country for this analysis.
    for preferred in ["origin", "country of origin", "country_origin", "source country"]:
        if preferred in col_map:
            return col_map[preferred]

    return pick_first_column(columns, ["origin", "country"])

all_columns = df.columns.tolist()
country_col = pick_country_column(all_columns)
date_col = pick_first_column(all_columns, ["date"])

reason_like_cols = [
    c for c in all_columns
    if any(k in c.lower() for k in [
        "reason", "ground", "basis", "subject", "description",
        "hazard", "non-compliance", "non compliance", "notification",
    ])
]

print("Detected country column:", country_col)
print("Detected date column:", date_col)
print("Detected reason-like columns:", reason_like_cols[:8], "..." if len(reason_like_cols) > 8 else "")

if country_col is None:
    raise ValueError("Could not detect a country/origin column. Please set country_col manually.")
if date_col is None:
    raise ValueError("Could not detect a date column. Please set date_col manually.")
if not reason_like_cols:
    raise ValueError("Could not detect reason-related columns. Please set reason_like_cols manually.")

df_tmp = df.copy()
df_tmp["_parsed_date"] = pd.to_datetime(
    df_tmp[date_col],
    errors="coerce",
    dayfirst=True,
    utc=False,
)

country_mask = df_tmp[country_col].astype(str).str.lower().str.strip().isin(TARGET_COUNTRIES)
date_mask = df_tmp["_parsed_date"].between(START_DATE, END_DATE, inclusive="both")
df_focus = df_tmp[country_mask & date_mask].copy()

def combine_reason_text(row: pd.Series, cols: list[str]) -> str:
    parts = [str(row[c]) for c in cols if pd.notna(row[c]) and str(row[c]).strip()]
    return " | ".join(parts)

df_focus["reason_text"] = df_focus.apply(lambda r: combine_reason_text(r, reason_like_cols), axis=1)

sps_pattern = re.compile("|".join(SPS_KEYWORDS), flags=re.IGNORECASE)
tbt_pattern = re.compile("|".join(TBT_KEYWORDS), flags=re.IGNORECASE)

def classify_reason(text: str) -> str:
    txt = str(text)
    has_sps = bool(sps_pattern.search(txt))
    has_tbt = bool(tbt_pattern.search(txt))
    if has_sps and has_tbt:
        return "SPS+TBT"
    if has_sps:
        return "SPS"
    if has_tbt:
        return "TBT"
    return "Other/Unmatched"

df_focus["category"] = df_focus["reason_text"].apply(classify_reason)

print(f"Validation date window: {START_DATE.date()} to {END_DATE.date()}")
print(f"Focused dataset rows: {len(df_focus):,}")
df_focus[[country_col, date_col, "category", "reason_text"]].head(10)

Detected country column: origin
Detected date column: date
Detected reason-like columns: ['subject', 'hazards'] 
Validation date window: 2020-03-30 to 2025-12-31
Focused dataset rows: 2,299


,origin,date,category,reason_text
1034,India,12-12-2025 16:13:55,SPS,Mineral oil residues (MOSH/MOAH) in rice from...
1049,India,12-12-2025 12:03:18,Other/Unmatched,Food supplement with novel food ingredient fro...
1062,India,11-12-2025 16:38:40,SPS,Pesticides residues in food supplement from In...
1067,Vietnam,11-12-2025 15:40:13,SPS,Oxytetracycline in frozen giant shrimp tails (...
1083,India,11-12-2025 11:19:22,Other/Unmatched,Pyrrolizidine alkaloids in black tea from Indi...
1084,India,11-12-2025 10:49:44,SPS,Nitrofurans ((SEM) in shrimps from India | nit...
1085,India,11-12-2025 10:37:10,SPS,chlorpyrifos in basmati rice from India | chlo...
1095,Vietnam,10-12-2025 15:36:07,SPS,Leucomalachite Green in Vannamei shrimps from ...
1096,Vietnam,10-12-2025 14:39:28,Other/Unmatched,Foreign body (metal wire) in ground black pepp...
1102,India,10-12-2025 12:06:08,SPS,Chlorpyrifos in masala spice from India. | chl...


In [9]:
# 1) Overall category distribution for selected countries
overall_summary = (
    df_focus["category"]
    .value_counts(dropna=False)
    .rename_axis("category")
    .reset_index(name="count")
)
print("Overall category distribution:")
print(overall_summary.to_string(index=False))

# 2) Country x category pivot
country_category_summary = (
    df_focus.groupby([country_col, "category"])
    .size()
    .reset_index(name="count")
    .sort_values([country_col, "count"], ascending=[True, False])
)
print("\nCountry-category detail:")
print(country_category_summary.to_string(index=False))

country_category_pivot = (
    country_category_summary
    .pivot(index=country_col, columns="category", values="count")
    .fillna(0)
    .astype(int)
)
print("\nCountry-category pivot table:")
print(country_category_pivot)

# 3) Sample unmatched rows for manual keyword refinement
unmatched_samples = df_focus[df_focus["category"] == "Other/Unmatched"][[country_col, "reason_text"]].head(20)
print("\nSample unmatched rows (for keyword tuning):")
print(unmatched_samples.to_string(index=False))

country_category_pivot

Overall category distribution:
       category  count
            SPS   1717
Other/Unmatched    390
            TBT    180
        SPS+TBT     12

Country-category detail:
 origin        category  count
Ecuador             SPS    125
Ecuador Other/Unmatched     39
Ecuador             TBT     27
Ecuador         SPS+TBT      1
  India             SPS   1341
  India Other/Unmatched    252
  India             TBT    119
  India         SPS+TBT      7
Vietnam             SPS    251
Vietnam Other/Unmatched     99
Vietnam             TBT     34
Vietnam         SPS+TBT      4

Country-category pivot table:
category  Other/Unmatched   SPS  SPS+TBT  TBT
origin                                       
Ecuador                39   125        1   27
India                 252  1341        7  119
Vietnam                99   251        4   34

Sample unmatched rows (for keyword tuning):
 origin                                                                                                                

category,Other/Unmatched,SPS,SPS+TBT,TBT
origin,,,,
Ecuador,39,125,1,27
India,252,1341,7,119
Vietnam,99,251,4,34


In [10]:
output_file = Path("EU_Vietnam_India_Ecuador_SPS_TBT.csv")
df_focus.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"Saved: {output_file.resolve()}")

Saved: K:\Khanhs\Study\Tutorial\Tutorial-2026\DE_Refusals_US_EU\KTPT\EU\EU_Vietnam_India_Ecuador_SPS_TBT.csv
